In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model(model="gemini-3-flash-preview",model_provider="google_genai")
standard_model = init_chat_model("gpt-5-nano")

@wrap_model_call
def state_based_model(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on state conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)

    if message_count > 10:
        model = large_model
    else:
        model = standard_model
    
    request = request.override(model=model)

    return handler(request)


In [14]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [15]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Did you water the office plant today?")
        ]
    }
)

print(response["messages"][-1].content)

Yes—I watered it this morning. If you’d like, I can set up a reminder for the next watering and keep a quick log of watering dates so we don’t miss any. Do you prefer a 3-day cycle, or a specific day of the week?


In [17]:
print(response["messages"][-1].response_metadata["model_name"])

gpt-5-nano-2025-08-07


In [5]:
from langchain.messages import AIMessage

response = agent.invoke(
    {
        "messages":[
            HumanMessage(content="Did you water the office plant today?"),
            AIMessage(content="Yes, I gave it a light watering this morning."),
            HumanMessage(content="Has it grown much this week?"),
            AIMessage(content="It's sprouted two new leaves since Monday."),
            HumanMessage(content="Are the leaves still turning yellow on the edges?"),
            AIMessage(content="A little, but it is looking healthier overall."),
            HumanMessage(content="Did you remember to rotate the pot towards the window?"),
            AIMessage(content="I rotated it a quarter turn, so it gets more even light."),
            HumanMessage(content="How often should we be fertilizing this plan?"),
            AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
            HumanMessage(content="When should we expect to have to replace the pot?")
        ]
    }
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "I looked that up earlier! Most plants like this only need a bigger pot every 12 to 18 months. I’ll keep an eye out for roots poking through the drainage holes at the bottom or if the soil starts drying out unusually fast—those are usually the big signs it's outgrown its home.\n\nI've made a note to check it again more closely in the spring, since that’s usually the best time of year to move it anyway! Does that sound like a good plan?", 'extras': {'signature': 'EqQOCqEOAb4+9vvFho7ea+yzCAQx0dRRjn8ZQVcCMtqmSvvPoGbFriz7Dy5YRbhm/vaq4IZ8d0E1uGUBdf3sxOKUdHtPByIb5+Htym/zNU8jPMJ4YFmY+V3jylhJOAKAMjToUAUMdpwRpCfaeWrd/FSz+vT/a0sbg5GppShFriZtrGdZBDKw57nNKZ8Od8el+eRKhJvQYN4suCtQIE+RByzWNGTxPMnLNEPv395NsXwatmaiPmQOcLn04eWTtrMpMkRoVIr9MpnX7Vmo94P4ghMjmx68jead5nq46et+pVL2mEvyD61uXmv6HWsqryhUdoLy14EilibS7cwfNhmKg5BEfjypdeV3j6gQg5ONl4S8vHzXGs6rdIAU7MacqR9Wg1EVdSfkEvq1dfYr1vMB15wERKfPxNWzNTOVa0hFAeFutrv+qumjs0Wa6114Q50HcOuWdL6E39txKOJjJi5ZaGPHFRd3wny7G+DHTCsMUMh9TluAGZDjPdWX6Kw

In [13]:
print(response["messages"][-1].response_metadata)

{'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}
